In [1]:
import sys
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "splits"

sys.path.insert(0, str(ROOT))
from src.preprocess import clean_tweet, preprocess

train_df = pd.read_csv(DATA / "train.csv")
train_df = preprocess(train_df)

print(train_df.shape)
train_df.head()

(200000, 4)


,label,id,tweet,clean_tweet
0,0,2323739350,Never ever!! Eat an ice cream sandwich out of ...,never ever!! eat an ice cream sandwich out of ...
1,0,2234329532,SA .. yet again,sa .. yet again
2,1,1834435622,@aussiemcflyfan aww really? maybe in a few yea...,USER aww really? maybe in a few years he will ...
3,0,2060875771,Im so tired!!! Baby sitting.,im so tired!!! baby sitting.
4,0,2189485033,"On my way to see about getting a bus, aren't c...","on my way to see about getting a bus, aren't c..."


In [2]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
example_text = "Transformers make NLP seamless!"
inputs = tokenizer(example_text)
print(inputs)
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"]))

{'input_ids': [101, 19081, 2191, 17953, 2361, 25180, 3238, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
['[CLS]', 'transformers', 'make', 'nl', '##p', 'seam', '##less', '!', '[SEP]']


In [3]:
texts = train_df["clean_tweet"].tolist()
encodings = tokenizer(texts)

token_counts = []
for ids in encodings["input_ids"]:
    token_counts.append(len(ids))

train_df["num_tokens"] = token_counts

train_df[["clean_tweet", "num_tokens"]].head()

,clean_tweet,num_tokens
0,never ever!! eat an ice cream sandwich out of ...,18
1,sa .. yet again,7
2,USER aww really? maybe in a few years he will ...,23
3,im so tired!!! baby sitting.,11
4,"on my way to see about getting a bus, aren't c...",22


In [4]:
train_token = train_df["num_tokens"]
print(train_token.describe())
print(train_token.quantile([0.95, 0.99]))
print("over 64 tokens:", (train_token > 64).sum())

longest_tweets = train_df.sort_values(by="num_tokens", ascending=False)
longest_tweets[["clean_tweet", "num_tokens"]].head(50)

count    200000.00000
mean         20.39665
std           9.81265
min           3.00000
25%          12.00000
50%          19.00000
75%          28.00000
max         163.00000
Name: num_tokens, dtype: float64
0.95    37.0
0.99    43.0
Name: num_tokens, dtype: float64
over 64 tokens: 124


,clean_tweet,num_tokens
61814,ê·¼ë?° ë¡ë°¥ì?´ìì¼ë ì± ìì§ê³ ë ë¤í...,163
72838,things â ñ?ñð¾ ððµññ! ð¡ñð¼ðµð» ñð°ð·ð...,160
98848,"ð?ññ, openoffice ð²ñð±ðµñ?ð¸ð» ñ?ð²ð¾ðµð¹ ð...",159
64117,ð²ñð±ðµð³ð°ð» ð½ð° ñð»ð¸ññ ñð°ð¼ ñð°ðºð°...,154
59428,"USER íë¦° ë§?ìì? ìëì­ëë¤ë§, íê...",150
84200,"ððµñññ ñð°ð·, ð¿ð°ð´ð°ðµññð°, ñð°ðº ð°...",147
152365,USER ñ? ð½ðµ ð¿ð¸ñ?ð°ð» ññð¾ ð¢ððð ð?ðð£...,146
120969,USER ð±ð¸ð·ð½ðµñ? ð½ðµ ð¸ð¼ðµðµñ ð³ñð°ð½ð¸ñ...,141
28515,"å¨ä¿¡ç¨å?¡ç""ç©ç?""å??å©ä¸, æ ¹æ?®æç¨...",137
4434,USER ñðµ ð²ð¸ð´ð¸ð¼ ðºð°ðº ñðµ ñð°ð·ð²ð¸ñ?ñ...,123


## token lengths with DistilBERT's tokenizer

To choose the maximum input length for fine-tuning, I tokenized all 200,000 cleaned training tweets with the `distilbert-base-uncased` tokenizer and counted the tokens per tweet, including the special `[CLS]` and `[SEP]` tokens.

### How the tokenizer splits text
The tokenizer splits words it doesn't know into known sub-word pieces, marked with `##`. For example, "Transformers make NLP seamless!" becomes `[CLS] transformers make nl ##p seam ##less ! [SEP]`.

count    200000.00000
mean         20.39665
std           9.81265
min           3.00000
25%          12.00000
50%          19.00000
75%          28.00000
max         163.00000
Name: num_tokens, dtype: float64
0.95    37.0
0.99    43.0
Name: num_tokens, dtype: float64
over 64 tokens: 124

### Results
| Statistic | Tokens |
|---|----|
| Median | 19 |
| 95th percentile | 37 |
| 99th percentile | 43 |
| Maximum | 163 |
| Tweets over 64 tokens | 124 of 200,000 (0.06%) |

The median tweet had 12 words. With 19 tokens at the median, including the two special tokens, that is about 17 tokens for 12 words, or roughly 1.4 tokens per word.

### What makes tweets long
- **Garbled non-English tweets.** The longest tweets (up to 163 tokens) are not english, such as Russian, Korean and Chinese. These tweets show an encoding error in the original file, producing strings like `ð²ñð±ðµ`. The scrambled characters match nothing in DistilBERT's mostly English vocabulary, so they split into many tiny tokens.
- **Repeated punctuation in English tweets.** The longest English tweets are long mainly because of runs like "!!!!!!!!!!" and stretched words like "wooowwww". The tokenizer makes each `!` a separate token, while my baseline's token pattern kept such a run as a single token.

### Decision
**`max_length = 64`.** Only 0.06% of tweets exceed it, mostly garbled non-English text. The shorter length keeps training fast.

The garbled tweets were kept, so that the transformer sees exactly the same data as the baseline.

## training platform
- **Kaggle notebooks**, chosen for a fixed weekly GPU quota and stable sessions.
- GPU: Tesla T4.
- The three splits and `preprocess.py` are uploaded as a Kaggle dataset.

## Part F: training plan
| Setting | Value |
|---|---|
| Model | `distilbert-base-uncased` |
| Preprocessing | `clean_tweet`, identical to the baseline |
| `max_length` | 64 |
| Training tweets | 50,000 test run, then all 200,000 |
| Epochs | 1–2 |
| Learning rate | 2e-5 |
| Batch size | 32 or 64 |
| Evaluation | validation after each epoch; test once at the end |

## Part G: summary of step 6
**Model:** `distilbert-base-uncased`, a general-purpose pretrained transformer, about 40% smaller and 60% faster than BERT. It is uncased, matching my lowercased preprocessing, and small enough to fine-tune on a free GPU.

**Alternatives considered:**
`bert-base-uncased` (slower, small gain), `roberta-base` (cased, slower), and the tweet-specific `vinai/bertweet-base` and `cardiffnlp/twitter-roberta-base-sentiment-latest`. The latter is already fine-tuned on other labeled tweets, so it would not be a fair comparison. A tweet-specific model is a possible optional second experiment.

**Fair comparison with the baseline:** same splits, same preprocessing (`clean_tweet`), same test set used once.